In [4]:
import os
import json
import glob
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, Image, clear_output

# ==========================================
# 1. CONFIGURATION
# ==========================================
BASE_PATH = "/pscratch/sd/t/tihsu/database/GridStudy_v2/method/"
JSON_GRID_PATH = "config/sample_list.json" 

# ==========================================
# 2. DATA LOADER
# ==========================================

def load_reference_grid(json_path):
    mx_set, my_set = set(), set()
    if os.path.exists(json_path):
        with open(json_path, 'r') as f:
            config = json.load(f)
        for mass_set in config.get("signal", []):
            match = re.search(r"MX-([\d\.]+)_MY-([\d\.]+)", mass_set)
            if match:
                mx_set.add(int(float(match.group(1))))
                my_set.add(int(float(match.group(2))))
    return sorted(list(mx_set)), sorted(list(my_set))

def parse_mass_robust(text):
    # Parses MX-AAA_MY-BBB, handling both int and float string formats
    match = re.search(r"MX-([\d\.]+).*?_MY-([\d\.]+)", text)
    if match:
        try:
            return int(float(match.group(1))), int(float(match.group(2)))
        except: return None, None
    return None, None

def load_grid_data(base_path):
    records = []

    def process_file(json_path, method, train_type):
        try:
            # 1. Parse coordinates
            # Priority: Filename -> Parent Folder -> Full Path
            mx, my = parse_mass_robust(os.path.basename(json_path))
            if mx is None:
                mx, my = parse_mass_robust(os.path.dirname(json_path).split(os.sep)[-1])
            if mx is None:
                mx, my = parse_mass_robust(json_path)
            
            if mx is None: return

            if "eval_metric" not in json_path: return

            # 2. Load Content
            with open(json_path, 'r') as f: data = json.load(f)

            # --- CRITICAL FIX ---
            # Explicitly check if 'auc' exists and is not NaN.
            # This prevents 'model.json' (which has no metrics) from overwriting 'metrics.json'.
            auc_val = data.get("auc", np.nan)
            if pd.isna(auc_val): 
                return

            # 3. Find PNG
            dirname = os.path.dirname(json_path)
            png_candidates = glob.glob(os.path.join(dirname, "*.png"))
            
            # Smart match: find PNG with specific mass ID first
            specific = [p for p in png_candidates if f"{mx}" in p and f"{my}" in p and "score" in p]
            if specific:
                png_path = specific[0]
            else:
                # Fallback to any score png
                png_path = next((p for p in png_candidates if "score" in p), None)

            records.append({
                "Method": method, "Type": train_type, "ID": f"{method} ({train_type})",
                "MX": int(mx), "MY": int(my),
                "auc": auc_val,
                "auc_unc": data.get("auc_unc", 0.0),
                "max_sic": data.get("max_sic", np.nan), 
                "max_sic_unc": data.get("max_sic_unc", 0.0),
                "time": data.get("fitting_time", np.nan),
                "png_path": png_path
            })
        except Exception as e:
            return

    # Recursive scan
    # 1. Individual
    for p in glob.glob(os.path.join(base_path, "*", "individual", "**", "*.json"), recursive=True):
        parts = p.split(os.sep)
        if "individual" in parts:
            # Extract method name which is parent of 'individual'
            method = parts[parts.index("individual")-1]
            process_file(p, method, "individual")

    # 2. Parameterized
    for p in glob.glob(os.path.join(base_path, "*", "parameterized", "**", "*.json"), recursive=True):
        parts = p.split(os.sep)
        if "parameterized" in parts:
            method = parts[parts.index("parameterized")-1]
            process_file(p, method, "parameterized")

    return pd.DataFrame(records)

# ==========================================
# 3. DASHBOARD CLASS
# ==========================================

class GridDashboard:
    def __init__(self):
        self.df = pd.DataFrame()
        self.expected_mx = []
        self.expected_my = []
        
        # UI
        self.btn_load = widgets.Button(description="Refresh Data", button_style='primary', icon='refresh')
        self.dd_metric = widgets.Dropdown(description="Metric:", options=[('AUC', 'auc'), ('Max SIC', 'max_sic'), ("Time", "time")], value='auc')
        self.tog_compare = widgets.ToggleButtons(options=['Ratio', 'Pull'], description='Compare:', button_style='')
        
        self.cb_show_text = widgets.Checkbox(value=True, description='Values')
        self.slider_font = widgets.IntSlider(value=10, min=6, max=18, description='Font:')
        self.slider_prec = widgets.IntSlider(value=3, min=0, max=5, description='Decimals:')
        
        self.dd_baseline = widgets.Dropdown(description="Baseline:")
        self.dd_mx = widgets.Dropdown(description="MX:")
        self.dd_my = widgets.Dropdown(description="MY:")
        
        self.out_status = widgets.Output()
        self.out_heatmap = widgets.Output()
        self.out_details = widgets.Output()
        
        # Events
        self.btn_load.on_click(self.load_data)
        for w in [self.dd_metric, self.dd_baseline, self.tog_compare, self.cb_show_text, self.slider_font, self.slider_prec]:
            w.observe(self.render_plots, names='value')
        self.dd_mx.observe(self.update_my_options, names='value')
        self.dd_my.observe(self.render_details, names='value')

        self.ui = widgets.VBox([
            widgets.HBox([self.btn_load, self.dd_metric, self.tog_compare]),
            widgets.HBox([self.dd_baseline, self.cb_show_text, self.slider_font, self.slider_prec]),
            self.out_status, widgets.HTML("<hr>"), self.out_heatmap,
            widgets.HTML("<hr><h3>Drill Down Inspector</h3>"), widgets.HBox([self.dd_mx, self.dd_my]), self.out_details
        ])
        self.load_data(None)

    def load_data(self, b):
        with self.out_status:
            clear_output()
            print("Loading...")
            ref_mx, ref_my = load_reference_grid(JSON_GRID_PATH)
            df_raw = load_grid_data(BASE_PATH)
            
            if not df_raw.empty:
                # Remove duplicates, ensuring we have data
                self.df = df_raw.sort_values('auc', ascending=False).drop_duplicates(subset=['ID', 'MX', 'MY'], keep='first')
                
                # Union axes: Config + Discovered
                self.expected_mx = sorted(list(set(ref_mx) | set(self.df['MX'].unique())))
                self.expected_my = sorted(list(set(ref_my) | set(self.df['MY'].unique())))
                print(f"Loaded {len(self.df)} records.")
            else:
                self.df = pd.DataFrame(columns=['ID', 'MX', 'MY', 'auc'])
                print("No data found.")

        if not self.df.empty:
            methods = sorted(self.df['ID'].unique())
            self.dd_baseline.options = methods
            if methods: self.dd_baseline.value = next((m for m in methods if 'base' in m.lower()), methods[0])
            self.dd_mx.options = self.expected_mx
            if self.expected_mx: 
                self.dd_mx.value = self.expected_mx[0]
                self.update_my_options(None)
        
        self.render_plots(None)

    def update_my_options(self, change):
        self.dd_my.options = self.expected_my
        if self.expected_my: self.dd_my.value = self.expected_my[0]
        self.render_details(None)

    def get_pivot(self, method, metric):
        sub = self.df[self.df['ID'] == method]
        if sub.empty: return pd.DataFrame(index=self.expected_my, columns=self.expected_mx)
        return sub.pivot_table(index='MY', columns='MX', values=metric, aggfunc='mean').reindex(index=self.expected_my, columns=self.expected_mx).sort_index(ascending=False)

    def render_plots(self, change):
        if not self.expected_mx or not self.dd_baseline.value: return
        with self.out_heatmap:
            clear_output(wait=True)
            met, base = self.dd_metric.value, self.dd_baseline.value
            methods = sorted(self.df['ID'].unique())
            n = len(methods)
            fig, axes = plt.subplots(n, 2, figsize=(16, 5*n), dpi=100)
            if n==1: axes = axes.reshape(1, -1)
            
            p_base = self.get_pivot(base, met)
            p_base_unc = self.get_pivot(base, f"{met}_unc")
            g_min, g_max = self.df[met].min(), self.df[met].max()

            for i, m in enumerate(methods):
                p_curr = self.get_pivot(m, met)
                
                # Plot 1: Absolute
                sns.heatmap(p_curr, ax=axes[i,0], annot=self.cb_show_text.value, fmt=f".{self.slider_prec.value}f", cmap="PuBu", vmin=g_min, vmax=g_max, cbar=True)
                axes[i,0].set_title(f"{m} {'(BASELINE)' if m==base else ''}", fontweight='bold')
                axes[i,0].set_ylabel("MY")
                
                # Plot 2: Compare
                ax_c = axes[i,1]
                if self.tog_compare.value == 'Ratio':
                    sns.heatmap(p_curr/p_base, ax=ax_c, annot=self.cb_show_text.value, fmt=f".{self.slider_prec.value}f", cmap="RdBu_r", center=1, vmin=0.5, vmax=1.5)
                    ax_c.set_title("Ratio (Method/Base)")
                else:
                    unc = np.sqrt(self.get_pivot(m, f"{met}_unc").fillna(0)**2 + p_base_unc.fillna(0)**2)
                    pull = (p_curr - p_base).divide(unc).replace([np.inf, -np.inf], np.nan)
                    sns.heatmap(pull, ax=ax_c, annot=self.cb_show_text.value, fmt=".1f", cmap="RdBu_r", center=0, vmin=-3, vmax=3)
                    ax_c.set_title("Pull (Sigma)")
                
                axes[i,0].set_xlabel("MX" if i==n-1 else "")
                ax_c.set_xlabel("MX" if i==n-1 else "")
                ax_c.set_yticks([])

            plt.tight_layout()
            plt.show()

    def render_details(self, change):
        mx, my, base = self.dd_mx.value, self.dd_my.value, self.dd_baseline.value
        if not mx or not my: return
        with self.out_details:
            clear_output(wait=True)
            display(widgets.HTML(f"<h4>Drill Down: {mx}, {my}</h4>"))
            
            html = "<table border='1' style='width:100%; border-collapse:collapse; text-align:center;'><tr><th>Method</th><th>AUC</th><th>Max SIC</th><th>Time</th></tr>"
            base_row = self.df[(self.df['ID']==base)&(self.df['MX']==mx)&(self.df['MY']==my)]
            base_sic = base_row.iloc[0]['max_sic'] if not base_row.empty else 0
            base_sic_unc = base_row.iloc[0]['max_sic_unc'] if not base_row.empty else 0

            imgs = []
            for m in sorted(self.df['ID'].unique()):
                row = self.df[(self.df['ID']==m)&(self.df['MX']==mx)&(self.df['MY']==my)]
                bg = "#eafaf1" if m==base else "white"
                
                if row.empty:
                    html += f"<tr style='background:{bg}'><td>{m}</td><td colspan='3' style='color:#aaa'>No Data</td></tr>"
                    imgs.append(widgets.HTML(f"<div style='border:1px solid #ccc; margin:5px; padding:20px; text-align:center; width:300px'><b>{m}</b><br>No Image</div>"))
                    continue
                
                r = row.iloc[0]
                
                # Pull Calculation
                sic_val, sic_unc = r['max_sic'], r.get('max_sic_unc', 0)
                comb_unc = np.sqrt(sic_unc**2 + base_sic_unc**2)
                pull = (sic_val - base_sic)/comb_unc if comb_unc>0 and m!=base else 0
                
                # Color logic
                color = "green" if pull > 2 else "red" if pull < -2 else "black"
                sic_str = f"{sic_val:.2f} ± {sic_unc:.2f}"
                if m != base and not pd.isna(base_sic): 
                    sic_str += f" <br><span style='color:{color}; font-size:0.8em'>({pull:.1f}σ)</span>"

                html += f"<tr style='background:{bg}'><td>{m}</td><td>{r['auc']:.4f}</td><td>{sic_str}</td><td>{r['time']:.1f}</td></tr>"
                
                img_w = widgets.HTML("<div style='color:#ccc'>Missing PNG</div>")
                if r['png_path'] and os.path.exists(r['png_path']):
                    img_w = widgets.Image(value=open(r['png_path'], "rb").read(), format='png', width=300)
                imgs.append(widgets.VBox([widgets.Label(m, style={'font-weight':'bold'}), img_w], layout=widgets.Layout(border='1px solid #eee', margin='5px', padding='5px')))

            html += "</table>"
            display(widgets.HTML(html))
            display(widgets.Box(imgs, layout=widgets.Layout(display='flex', flex_flow='row wrap')))

d = GridDashboard()
display(d.ui)

In [6]:
import os
import json
import glob
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, Image, clear_output

# ==========================================
# 1. CONFIGURATION
# ==========================================
BASE_PATH = "/pscratch/sd/t/tihsu/database/GridStudy_v2/method/"
JSON_GRID_PATH = "config/sample_list.json" 
CUTFLOW_PATH = "/pscratch/sd/t/tihsu/database/GridStudy/cutflow.txt"

# ==========================================
# 2. DATA LOADER
# ==========================================

def load_reference_grid(json_path):
    mx_set, my_set = set(), set()
    if os.path.exists(json_path):
        with open(json_path, 'r') as f:
            config = json.load(f)
        for mass_set in config.get("signal", []):
            match = re.search(r"MX-([\d\.]+)_MY-([\d\.]+)", mass_set)
            if match:
                mx_set.add(int(float(match.group(1))))
                my_set.add(int(float(match.group(2))))
    return sorted(list(mx_set)), sorted(list(my_set))

def parse_mass_robust(text):
    match = re.search(r"MX-([\d\.]+).*?_MY-([\d\.]+)", text)
    if match:
        try:
            return int(float(match.group(1))), int(float(match.group(2)))
        except: return None, None
    return None, None

def load_cutflow_data(path):
    """Parses cutflow.txt into a DataFrame for plotting."""
    records = []
    if not os.path.exists(path):
        print(f"Warning: Cutflow file not found at {path}")
        return pd.DataFrame()
        
    try:
        with open(path, 'r') as f:
            for line in f:
                if "MX-" not in line or "|" not in line: continue
                parts = [p.strip() for p in line.split('|')]
                
                # Expected format: Name | Total | Passed | Eff
                if len(parts) >= 4:
                    mx, my = parse_mass_robust(parts[0])
                    if mx is not None:
                        records.append({
                            "MX": mx, 
                            "MY": my,
                            "passed": float(parts[2]), # Passed count
                            "eff": float(parts[3])     # Efficiency %
                        })
    except Exception as e:
        print(f"Error parsing cutflow: {e}")
        
    return pd.DataFrame(records)

def load_grid_data(base_path):
    records = []

    def process_file(json_path, method, train_type):
        try:
            mx, my = parse_mass_robust(os.path.basename(json_path))
            if mx is None: mx, my = parse_mass_robust(os.path.dirname(json_path).split(os.sep)[-1])
            if mx is None: mx, my = parse_mass_robust(json_path)
            if mx is None: return

            if "eval_metric" not in json_path: return

            with open(json_path, 'r') as f: data = json.load(f)
            auc_val = data.get("auc", np.nan)
            if pd.isna(auc_val): return

            dirname = os.path.dirname(json_path)
            png_candidates = glob.glob(os.path.join(dirname, "*.png"))
            specific = [p for p in png_candidates if f"{mx}" in p and f"{my}" in p and "score" in p]
            png_path = specific[0] if specific else next((p for p in png_candidates if "score" in p), None)

            records.append({
                "Method": method, "Type": train_type, "ID": f"{method} ({train_type})",
                "MX": int(mx), "MY": int(my),
                "auc": auc_val,
                "auc_unc": data.get("auc_unc", 0.0),
                "max_sic": data.get("max_sic", np.nan), 
                "max_sic_unc": data.get("max_sic_unc", 0.0),
                "time": data.get("fitting_time", np.nan),
                "png_path": png_path
            })
        except: return

    for p in glob.glob(os.path.join(base_path, "*", "individual", "**", "*.json"), recursive=True):
        parts = p.split(os.sep)
        if "individual" in parts: process_file(p, parts[parts.index("individual")-1], "individual")

    for p in glob.glob(os.path.join(base_path, "*", "parameterized", "**", "*.json"), recursive=True):
        parts = p.split(os.sep)
        if "parameterized" in parts: process_file(p, parts[parts.index("parameterized")-1], "parameterized")

    return pd.DataFrame(records)

# ==========================================
# 3. DASHBOARD CLASS
# ==========================================

class GridDashboard:
    def __init__(self):
        self.df = pd.DataFrame()
        self.df_cutflow = pd.DataFrame() # Store cutflow data
        self.expected_mx = []
        self.expected_my = []
        
        # UI Elements
        self.btn_load = widgets.Button(description="Refresh Data", button_style='primary', icon='refresh')
        
        # Added Cutflow options to the dropdown
        self.dd_metric = widgets.Dropdown(
            description="Metric:", 
            options=[
                ('AUC', 'auc'), 
                ('Max SIC', 'max_sic'), 
                ("Time", "time"),
                ('Cutflow: Passed', 'passed'), # New
                ('Cutflow: Eff %', 'eff')      # New
            ], 
            value='auc'
        )
        
        self.tog_compare = widgets.ToggleButtons(options=['Ratio', 'Pull'], description='Compare:', button_style='')
        self.cb_show_text = widgets.Checkbox(value=True, description='Values')
        self.slider_font = widgets.IntSlider(value=10, min=6, max=18, description='Font:')
        self.slider_prec = widgets.IntSlider(value=3, min=0, max=5, description='Decimals:')
        
        self.dd_baseline = widgets.Dropdown(description="Baseline:")
        self.dd_mx = widgets.Dropdown(description="MX:")
        self.dd_my = widgets.Dropdown(description="MY:")
        
        self.out_status = widgets.Output()
        self.out_heatmap = widgets.Output()
        self.out_details = widgets.Output()
        
        # Events
        self.btn_load.on_click(self.load_data)
        for w in [self.dd_metric, self.dd_baseline, self.tog_compare, self.cb_show_text, self.slider_font, self.slider_prec]:
            w.observe(self.render_plots, names='value')
        self.dd_mx.observe(self.update_my_options, names='value')
        self.dd_my.observe(self.render_details, names='value')

        self.ui = widgets.VBox([
            widgets.HBox([self.btn_load, self.dd_metric, self.tog_compare]),
            widgets.HBox([self.dd_baseline, self.cb_show_text, self.slider_font, self.slider_prec]),
            self.out_status, widgets.HTML("<hr>"), self.out_heatmap,
            widgets.HTML("<hr><h3>Drill Down Inspector</h3>"), widgets.HBox([self.dd_mx, self.dd_my]), self.out_details
        ])
        self.load_data(None)

    def load_data(self, b):
        with self.out_status:
            clear_output()
            print("Loading...")
            ref_mx, ref_my = load_reference_grid(JSON_GRID_PATH)
            
            # Load Cutflow
            self.df_cutflow = load_cutflow_data(CUTFLOW_PATH)
            if not self.df_cutflow.empty:
                print(f"Loaded Cutflow info for {len(self.df_cutflow)} signals.")
                
            df_raw = load_grid_data(BASE_PATH)
            
            if not df_raw.empty:
                self.df = df_raw.sort_values('auc', ascending=False).drop_duplicates(subset=['ID', 'MX', 'MY'], keep='first')
                self.expected_mx = sorted(list(set(ref_mx) | set(self.df['MX'].unique())))
                self.expected_my = sorted(list(set(ref_my) | set(self.df['MY'].unique())))
                print(f"Loaded {len(self.df)} records.")
            else:
                self.df = pd.DataFrame(columns=['ID', 'MX', 'MY', 'auc'])
                print("No data found.")

        if not self.df.empty:
            methods = sorted(self.df['ID'].unique())
            self.dd_baseline.options = methods
            if methods: self.dd_baseline.value = next((m for m in methods if 'base' in m.lower()), methods[0])
            self.dd_mx.options = self.expected_mx
            if self.expected_mx: 
                self.dd_mx.value = self.expected_mx[0]
                self.update_my_options(None)
        
        self.render_plots(None)

    def update_my_options(self, change):
        self.dd_my.options = self.expected_my
        if self.expected_my: self.dd_my.value = self.expected_my[0]
        self.render_details(None)

    def get_pivot(self, method, metric):
        sub = self.df[self.df['ID'] == method]
        if sub.empty: return pd.DataFrame(index=self.expected_my, columns=self.expected_mx)
        return sub.pivot_table(index='MY', columns='MX', values=metric, aggfunc='mean').reindex(index=self.expected_my, columns=self.expected_mx).sort_index(ascending=False)

    def render_plots(self, change):
        if not self.expected_mx: return
        with self.out_heatmap:
            clear_output(wait=True)
            met = self.dd_metric.value
            
            # --- 2D CUTFLOW PLOT LOGIC ---
            if met in ['passed', 'eff']:
                if self.df_cutflow.empty:
                    print("No Cutflow data available to plot.")
                    return
                
                # Create Pivot
                pivot = self.df_cutflow.pivot(index='MY', columns='MX', values=met).reindex(index=self.expected_my, columns=self.expected_mx).sort_index(ascending=False)
                
                fig, ax = plt.subplots(figsize=(10, 8), dpi=100)
                fmt = ".0f" if met == 'passed' else ".2f"
                sns.heatmap(pivot, ax=ax, annot=self.cb_show_text.value, fmt=fmt, cmap="viridis", cbar=True)
                
                title_str = "Number of Passed Events" if met == 'passed' else "Signal Efficiency (%)"
                ax.set_title(f"Cutflow Distribution: {title_str}", fontsize=14, fontweight='bold')
                ax.set_xlabel("MX")
                ax.set_ylabel("MY")
                plt.tight_layout()
                plt.show()
                return
            # -----------------------------

            if not self.dd_baseline.value: return
            base = self.dd_baseline.value
            methods = sorted(self.df['ID'].unique())
            n = len(methods)
            fig, axes = plt.subplots(n, 2, figsize=(16, 5*n), dpi=100)
            if n==1: axes = axes.reshape(1, -1)
            
            p_base = self.get_pivot(base, met)
            p_base_unc = self.get_pivot(base, f"{met}_unc")
            g_min, g_max = self.df[met].min(), self.df[met].max()

            for i, m in enumerate(methods):
                p_curr = self.get_pivot(m, met)
                
                # Plot 1: Absolute
                sns.heatmap(p_curr, ax=axes[i,0], annot=self.cb_show_text.value, fmt=f".{self.slider_prec.value}f", cmap="PuBu", vmin=g_min, vmax=g_max, cbar=True)
                axes[i,0].set_title(f"{m} {'(BASELINE)' if m==base else ''}", fontweight='bold')
                axes[i,0].set_ylabel("MY")
                
                # Plot 2: Compare
                ax_c = axes[i,1]
                if self.tog_compare.value == 'Ratio':
                    sns.heatmap(p_curr/p_base, ax=ax_c, annot=self.cb_show_text.value, fmt=f".{self.slider_prec.value}f", cmap="RdBu_r", center=1, vmin=0.5, vmax=1.5)
                    ax_c.set_title("Ratio (Method/Base)")
                else:
                    unc = np.sqrt(self.get_pivot(m, f"{met}_unc").fillna(0)**2 + p_base_unc.fillna(0)**2)
                    pull = (p_curr - p_base).divide(unc).replace([np.inf, -np.inf], np.nan)
                    sns.heatmap(pull, ax=ax_c, annot=self.cb_show_text.value, fmt=".1f", cmap="RdBu_r", center=0, vmin=-3, vmax=3)
                    ax_c.set_title("Pull (Sigma)")
                
                axes[i,0].set_xlabel("MX" if i==n-1 else "")
                ax_c.set_xlabel("MX" if i==n-1 else "")
                ax_c.set_yticks([])

            plt.tight_layout()
            plt.show()

    def render_details(self, change):
        mx, my, base = self.dd_mx.value, self.dd_my.value, self.dd_baseline.value
        if not mx or not my: return
        with self.out_details:
            clear_output(wait=True)
            
            # --- INFO PANEL ---
            cut_val = "N/A"
            if not self.df_cutflow.empty:
                subset = self.df_cutflow[(self.df_cutflow['MX']==mx) & (self.df_cutflow['MY']==my)]
                if not subset.empty:
                    cut_val = f"{int(subset.iloc[0]['passed'])} (Eff: {subset.iloc[0]['eff']}%)"
            
            display(widgets.HTML(f"""
            <div style="background-color: #e3f2fd; border-left: 6px solid #2196f3; padding: 10px; margin-bottom: 20px;">
                <h4 style="margin:0; color:#0d47a1;">Signal Point: MX-{mx} MY-{my}</h4>
                Passed Events: <b>{cut_val}</b>
            </div>
            """))
            # ------------------
            
            html = "<table border='1' style='width:100%; border-collapse:collapse; text-align:center;'><tr><th>Method</th><th>AUC</th><th>Max SIC</th><th>Time</th></tr>"
            base_row = self.df[(self.df['ID']==base)&(self.df['MX']==mx)&(self.df['MY']==my)]
            base_sic = base_row.iloc[0]['max_sic'] if not base_row.empty else 0
            base_sic_unc = base_row.iloc[0]['max_sic_unc'] if not base_row.empty else 0

            imgs = []
            for m in sorted(self.df['ID'].unique()):
                row = self.df[(self.df['ID']==m)&(self.df['MX']==mx)&(self.df['MY']==my)]
                bg = "#eafaf1" if m==base else "white"
                
                if row.empty:
                    html += f"<tr style='background:{bg}'><td>{m}</td><td colspan='3' style='color:#aaa'>No Data</td></tr>"
                    imgs.append(widgets.HTML(f"<div style='border:1px solid #ccc; margin:5px; padding:20px; text-align:center; width:300px'><b>{m}</b><br>No Image</div>"))
                    continue
                
                r = row.iloc[0]
                sic_val, sic_unc = r['max_sic'], r.get('max_sic_unc', 0)
                comb_unc = np.sqrt(sic_unc**2 + base_sic_unc**2)
                pull = (sic_val - base_sic)/comb_unc if comb_unc>0 and m!=base else 0
                color = "green" if pull > 2 else "red" if pull < -2 else "black"
                sic_str = f"{sic_val:.2f} ± {sic_unc:.2f}"
                if m != base and not pd.isna(base_sic): 
                    sic_str += f" <br><span style='color:{color}; font-size:0.8em'>({pull:.1f}σ)</span>"

                html += f"<tr style='background:{bg}'><td>{m}</td><td>{r['auc']:.4f}</td><td>{sic_str}</td><td>{r['time']:.1f}</td></tr>"
                
                img_w = widgets.HTML("<div style='color:#ccc'>Missing PNG</div>")
                if r['png_path'] and os.path.exists(r['png_path']):
                    img_w = widgets.Image(value=open(r['png_path'], "rb").read(), format='png', width=300)
                imgs.append(widgets.VBox([widgets.Label(m, style={'font-weight':'bold'}), img_w], layout=widgets.Layout(border='1px solid #eee', margin='5px', padding='5px')))

            html += "</table>"
            display(widgets.HTML(html))
            display(widgets.Box(imgs, layout=widgets.Layout(display='flex', flex_flow='row wrap')))

d = GridDashboard()
display(d.ui)

In [7]:
import os
import json
import glob
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import ipywidgets as widgets
import random
from IPython.display import display, Image, clear_output

# ==========================================
# 1. CONFIGURATION
# ==========================================
BASE_PATH = "/pscratch/sd/t/tihsu/database/GridStudy_v2/method/"
JSON_GRID_PATH = "config/sample_list.json" 
CUTFLOW_PATH = "/pscratch/sd/t/tihsu/database/GridStudy/cutflow.txt"

# ==========================================
# 2. DATA LOADER
# ==========================================

def load_reference_grid(json_path):
    mx_set, my_set = set(), set()
    if os.path.exists(json_path):
        with open(json_path, 'r') as f:
            config = json.load(f)
        for mass_set in config.get("signal", []):
            match = re.search(r"MX-([\d\.]+)_MY-([\d\.]+)", mass_set)
            if match:
                mx_set.add(int(float(match.group(1))))
                my_set.add(int(float(match.group(2))))
    return sorted(list(mx_set)), sorted(list(my_set))

def parse_mass_robust(text):
    match = re.search(r"MX-([\d\.]+).*?_MY-([\d\.]+)", text)
    if match:
        try:
            return int(float(match.group(1))), int(float(match.group(2)))
        except: return None, None
    return None, None

def load_cutflow_data(path):
    records = []
    if not os.path.exists(path):
        return pd.DataFrame()
        
    try:
        with open(path, 'r') as f:
            for line in f:
                if "MX-" not in line or "|" not in line: continue
                parts = [p.strip() for p in line.split('|')]
                if len(parts) >= 4:
                    mx, my = parse_mass_robust(parts[0])
                    if mx is not None:
                        records.append({
                            "MX": mx, 
                            "MY": my,
                            "passed": float(parts[2]),
                            "eff": float(parts[3])
                        })
    except Exception as e:
        print(f"Error parsing cutflow: {e}")
        
    return pd.DataFrame(records)

def load_grid_data(base_path):
    records = []

    def process_file(json_path, method, train_type):
        try:
            filename = os.path.basename(json_path)
            mx, my = parse_mass_robust(filename)
            if mx is None: mx, my = parse_mass_robust(os.path.dirname(json_path).split(os.sep)[-1])
            if mx is None: return

            if "eval_metric" not in json_path: return

            with open(json_path, 'r') as f: data = json.load(f)
            auc_val = data.get("auc", np.nan)
            if pd.isna(auc_val): return

            dirname = os.path.dirname(json_path)
            png_candidates = glob.glob(os.path.join(dirname, "*.png"))
            specific = [p for p in png_candidates if f"{mx}" in p and f"{my}" in p and "score" in p]
            png_path = specific[0] if specific else next((p for p in png_candidates if "score" in p), None)

            records.append({
                "Method": method, 
                "Type": train_type, 
                "ID": f"{method} ({train_type})",
                "MX": int(mx), "MY": int(my),
                "auc": auc_val,
                "auc_unc": data.get("auc_unc", 0.0),
                "max_sic": data.get("max_sic", np.nan), 
                "max_sic_unc": data.get("max_sic_unc", 0.0),
                "time": data.get("fitting_time", np.nan),
                "png_path": png_path
            })
        except Exception as e:
            return

    # 1. Individual
    for p in glob.glob(os.path.join(base_path, "*", "individual", "**", "*.json"), recursive=True):
        parts = p.split(os.sep)
        if "individual" in parts: 
            process_file(p, parts[parts.index("individual")-1], "individual")

    # 2. Parameterized
    for p in glob.glob(os.path.join(base_path, "*", "parameterized", "**", "*.json"), recursive=True):
        parts = p.split(os.sep)
        if "parameterized" in parts: 
            process_file(p, parts[parts.index("parameterized")-1], "parameterized")

    # 3. Sparse Parameterized
    sparse_pattern = os.path.join(base_path, "*", "parametrized_reduce_*", "All", "*.json")
    for p in glob.glob(sparse_pattern):
        parts = p.split(os.sep)
        try:
            method_name = parts[-4] 
            config_folder = parts[-3]
            factor_match = re.search(r"factor_x_(\d+)_y_(\d+)", config_folder)
            if factor_match:
                nx, ny = factor_match.groups()
                train_type = f"sparse_x{nx}y{ny}"
            else:
                train_type = "sparse_param"
            process_file(p, method_name, train_type)
        except IndexError:
            continue

    return pd.DataFrame(records)

# ==========================================
# 3. FANCY PLOTTING FUNCTION
# ==========================================

def plot_comparison_fancy(df, df_cutflow, target_mx, target_my, all_mx, all_my, metric='max_sic', baseline_id='xgboost (individual)'):
    """
    Generates the detailed grouping plot comparing models vs baseline with jitter.
    """
    # 1. Filter Data
    subset = df[(df['MX'] == target_mx) & (df['MY'] == target_my)].copy()
    if subset.empty:
        print("No data for this point.")
        return

    # 2. Get Baseline
    base_row = subset[subset['ID'] == baseline_id]
    if base_row.empty:
        fallback = [x for x in subset['ID'].unique() if 'xgboost' in x.lower() and 'individual' in x.lower()]
        if fallback:
            baseline_id = fallback[0]
            base_row = subset[subset['ID'] == baseline_id]
        else:
            print(f"Baseline {baseline_id} not found.")
            return

    base_val = base_row.iloc[0][metric]
    
    # 3. Get Sample Size (for Color)
    cut_val = 100 
    if not df_cutflow.empty:
        c_sub = df_cutflow[(df_cutflow['MX'] == target_mx) & (df_cutflow['MY'] == target_my)]
        if not c_sub.empty:
            cut_val = c_sub.iloc[0]['passed']

    # 4. Prepare Plot Data
    plot_data = []
    
    subset['ModelName'] = subset['Method']
    subset['ModeName'] = subset['Type']
    
    # Custom sorting: XGBoost first, then alphabetical
    models = sorted(subset['ModelName'].unique(), key=lambda x: (0 if 'xgboost' in x.lower() else 1, x))
    
    for model in models:
        # We might have duplicates if multiple runs exist, so we keep them to show scatter
        m_rows = subset[subset['ModelName'] == model].sort_values('ModeName')
        
        # Group by 'ModeName' (e.g. sparse_x4y4) so we can stack them on one x-tick
        modes = sorted(m_rows['ModeName'].unique())
        
        group_modes = []
        for mode in modes:
            mode_rows = m_rows[m_rows['ModeName'] == mode]
            
            points_info = []
            for _, row in mode_rows.iterrows():
                val = row[metric]
                ratio = val / base_val if base_val != 0 else 0
                
                # Training vs Interpolation Logic
                is_trained = False
                if 'individual' in row['ModeName'].lower():
                    is_trained = True
                elif 'sparse' in row['ModeName']:
                    match = re.search(r"x(\d+)y(\d+)", row['ModeName'])
                    if match:
                        step_x, step_y = int(match.group(1)), int(match.group(2))
                        try:
                            idx_x, idx_y = all_mx.index(target_mx), all_my.index(target_my)
                            if (idx_x % step_x == 0) and (idx_y % step_y == 0):
                                is_trained = True
                        except ValueError: pass
                
                points_info.append({
                    'ratio': ratio,
                    'is_trained': is_trained,
                    'sample_size': cut_val
                })
            
            group_modes.append({'mode_label': mode, 'points': points_info})
            
        plot_data.append({'model': model, 'modes': group_modes})

    # 5. Plotting
    fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
    cmap = sns.cubehelix_palette(start=2, rot=0, dark=0.2, light=.8, as_cmap=True)
    norm = plt.Normalize(vmin=0, vmax=cut_val*1.5)
    
    x_cursor = 0
    x_ticks = []
    x_labels = []
    
    # Jitter width
    jitter_width = 0.15 
    
    for group in plot_data:
        model_name = group['model']
        modes = group['modes']
        
        start_x = x_cursor
        
        for m_obj in modes:
            points = m_obj['points']
            
            # Add some noise to x based on number of points or just random
            # If there is only 1 point, x is centered. If multiple, spread them.
            
            for i, p in enumerate(points):
                # Deterministic jitter based on index so it redraws same way
                # (or use random.uniform(-jitter_width, jitter_width))
                noise = random.uniform(-jitter_width, jitter_width) if len(points) > 1 else 0
                
                marker = 'D' if p['is_trained'] else 'o'
                
                ax.scatter(x_cursor + noise, p['ratio'], 
                       c=[p['sample_size']], 
                       s=100, 
                       marker=marker, 
                       cmap=cmap, norm=norm, 
                       edgecolors='black', linewidth=0.5, zorder=10)
            
            x_ticks.append(x_cursor)
            lbl = m_obj['mode_label'].replace('sparse_', '').replace('individual', 'Indiv')
            # Shorten labels further if needed
            lbl = lbl.replace('factor_', '').replace('reduce_', '')
            x_labels.append(lbl)
            
            x_cursor += 1
            
        end_x = x_cursor - 1
        center_x = (start_x + end_x) / 2
        
        # Label Model
        ax.text(center_x, ax.get_ylim()[1] if ax.get_ylim()[1] > 1.2 else 1.2, 
                model_name.upper(), 
                ha='center', va='bottom', fontweight='bold', fontsize=11, color='#333')
        
        # Separator
        ax.axvline(x=x_cursor - 0.5, color='#eee', linestyle='-', zorder=0)
        
        # Gap between models
        x_cursor += 0.5

    ax.axhline(1.0, color='gray', linestyle='--', linewidth=1, zorder=0)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=9)
    
    ax.set_ylabel(f"Ratio to Baseline ({metric})", fontsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    legend_elements = [
        plt.Line2D([0], [0], marker='D', color='w', label='Trained Point',
                   markerfacecolor='gray', markersize=8, markeredgecolor='k'),
        plt.Line2D([0], [0], marker='o', color='w', label='Interpolated',
                   markerfacecolor='gray', markersize=8, markeredgecolor='k')
    ]
    ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1, 1), frameon=False, fontsize=9)
    
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, aspect=30, pad=0.02)
    cbar.set_label('Sample Size (Events)', rotation=270, labelpad=15)
    
    plt.title(f"Performance Landscape @ MX-{target_mx} MY-{target_my}", loc='left', fontsize=12, pad=20)
    plt.tight_layout()
    plt.show()

# ==========================================
# 4. DASHBOARD CLASS
# ==========================================

class GridDashboard:
    def __init__(self):
        self.df = pd.DataFrame()
        self.df_cutflow = pd.DataFrame()
        self.expected_mx = []
        self.expected_my = []
        
        # UI Elements
        self.btn_load = widgets.Button(description="Refresh Data", button_style='primary', icon='refresh')
        
        self.dd_metric = widgets.Dropdown(
            description="Metric:", 
            options=[('AUC', 'auc'), ('Max SIC', 'max_sic'), ("Time", "time"), ('Cutflow: Passed', 'passed'), ('Cutflow: Eff %', 'eff')], 
            value='auc'
        )
        
        self.tog_compare = widgets.ToggleButtons(options=['Ratio', 'Pull'], description='Compare:', button_style='')
        self.cb_show_text = widgets.Checkbox(value=True, description='Values')
        self.slider_font = widgets.IntSlider(value=10, min=6, max=18, description='Font:')
        self.slider_prec = widgets.IntSlider(value=3, min=0, max=5, description='Decimals:')
        
        self.dd_baseline = widgets.Dropdown(description="Baseline:")
        self.dd_mx = widgets.Dropdown(description="MX:")
        self.dd_my = widgets.Dropdown(description="MY:")
        
        self.out_status = widgets.Output()
        self.out_heatmap = widgets.Output()
        self.out_details = widgets.Output()
        
        # Events
        self.btn_load.on_click(self.load_data)
        for w in [self.dd_metric, self.dd_baseline, self.tog_compare, self.cb_show_text, self.slider_font, self.slider_prec]:
            w.observe(self.render_plots, names='value')
        self.dd_mx.observe(self.update_my_options, names='value')
        self.dd_my.observe(self.render_details, names='value')

        self.ui = widgets.VBox([
            widgets.HBox([self.btn_load, self.dd_metric, self.tog_compare]),
            widgets.HBox([self.dd_baseline, self.cb_show_text, self.slider_font, self.slider_prec]),
            self.out_status, widgets.HTML("<hr>"), self.out_heatmap,
            widgets.HTML("<hr><h3>Drill Down Inspector</h3>"), widgets.HBox([self.dd_mx, self.dd_my]), self.out_details
        ])
        self.load_data(None)

    def load_data(self, b):
        with self.out_status:
            clear_output()
            print("Loading...")
            ref_mx, ref_my = load_reference_grid(JSON_GRID_PATH)
            
            self.df_cutflow = load_cutflow_data(CUTFLOW_PATH)
            
            df_raw = load_grid_data(BASE_PATH)
            
            if not df_raw.empty:
                self.df = df_raw.sort_values('auc', ascending=False).drop_duplicates(subset=['ID', 'MX', 'MY'], keep='first')
                self.expected_mx = sorted(list(set(ref_mx) | set(self.df['MX'].unique())))
                self.expected_my = sorted(list(set(ref_my) | set(self.df['MY'].unique())))
                print(f"Loaded {len(self.df)} records.")
            else:
                self.df = pd.DataFrame(columns=['ID', 'MX', 'MY', 'auc'])
                print("No data found.")

        if not self.df.empty:
            methods = sorted(self.df['ID'].unique())
            self.dd_baseline.options = methods
            if methods: self.dd_baseline.value = next((m for m in methods if 'xgboost' in m.lower() and 'individual' in m.lower()), methods[0])
            self.dd_mx.options = self.expected_mx
            if self.expected_mx: 
                self.dd_mx.value = self.expected_mx[0]
                self.update_my_options(None)
        
        self.render_plots(None)

    def update_my_options(self, change):
        self.dd_my.options = self.expected_my
        if self.expected_my: self.dd_my.value = self.expected_my[0]
        self.render_details(None)

    def get_pivot(self, method, metric):
        sub = self.df[self.df['ID'] == method]
        if sub.empty: return pd.DataFrame(index=self.expected_my, columns=self.expected_mx)
        return sub.pivot_table(index='MY', columns='MX', values=metric, aggfunc='mean').reindex(index=self.expected_my, columns=self.expected_mx).sort_index(ascending=False)

    def render_plots(self, change):
        if not self.expected_mx: return
        with self.out_heatmap:
            clear_output(wait=True)
            met = self.dd_metric.value
            if met in ['passed', 'eff']:
                if self.df_cutflow.empty:
                    print("No Cutflow data.")
                    return
                pivot = self.df_cutflow.pivot(index='MY', columns='MX', values=met).reindex(index=self.expected_my, columns=self.expected_mx).sort_index(ascending=False)
                fig, ax = plt.subplots(figsize=(10, 8), dpi=100)
                sns.heatmap(pivot, ax=ax, annot=self.cb_show_text.value, fmt=".0f" if met=='passed' else ".2f", cmap="viridis", cbar=True)
                plt.show()
                return

            if not self.dd_baseline.value: return
            base = self.dd_baseline.value
            methods = sorted(self.df['ID'].unique())
            n = len(methods)
            fig, axes = plt.subplots(n, 2, figsize=(16, 5*n), dpi=100)
            if n==1: axes = axes.reshape(1, -1)
            
            p_base = self.get_pivot(base, met)
            g_min, g_max = self.df[met].min(), self.df[met].max()

            for i, m in enumerate(methods):
                p_curr = self.get_pivot(m, met)
                sns.heatmap(p_curr, ax=axes[i,0], annot=self.cb_show_text.value, fmt=f".{self.slider_prec.value}f", cmap="PuBu", vmin=g_min, vmax=g_max, cbar=True)
                axes[i,0].set_title(f"{m}", fontweight='bold')
                axes[i,0].set_ylabel("MY")
                
                ax_c = axes[i,1]
                if self.tog_compare.value == 'Ratio':
                    sns.heatmap(p_curr/p_base, ax=ax_c, annot=self.cb_show_text.value, fmt=f".{self.slider_prec.value}f", cmap="RdBu_r", center=1, vmin=0.5, vmax=1.5)
                    ax_c.set_title("Ratio")
                else:
                    sns.heatmap(p_curr - p_base, ax=ax_c, annot=self.cb_show_text.value, fmt=".2f", cmap="RdBu_r", center=0)
                    ax_c.set_title("Diff")
                axes[i,0].set_xlabel("MX" if i==n-1 else "")
                ax_c.set_xlabel("MX" if i==n-1 else "")
                ax_c.set_yticks([])

            plt.tight_layout()
            plt.show()

    def render_details(self, change):
        mx, my = self.dd_mx.value, self.dd_my.value
        base = self.dd_baseline.value
        if not mx or not my: return
        with self.out_details:
            clear_output(wait=True)
            
            cut_val = "N/A"
            if not self.df_cutflow.empty:
                subset = self.df_cutflow[(self.df_cutflow['MX']==mx) & (self.df_cutflow['MY']==my)]
                if not subset.empty:
                    cut_val = f"{int(subset.iloc[0]['passed'])} (Eff: {subset.iloc[0]['eff']}%)"
            
            display(widgets.HTML(f"""
            <div style="background-color: #e3f2fd; border-left: 6px solid #2196f3; padding: 10px; margin-bottom: 20px;">
                <h4 style="margin:0; color:#0d47a1;">Signal Point: MX-{mx} MY-{my}</h4>
                Passed Events: <b>{cut_val}</b>
            </div>
            """))

            plot_metric = 'max_sic' if 'sic' in self.dd_metric.value else 'auc'
            plot_base = next((m for m in self.df['ID'].unique() if 'xgboost' in m.lower() and 'individual' in m.lower()), base)
            
            try:
                plot_comparison_fancy(self.df, self.df_cutflow, mx, my, self.expected_mx, self.expected_my, metric=plot_metric, baseline_id=plot_base)
            except Exception as e:
                print(f"Could not render summary plot: {e}")

            html = "<table border='1' style='width:100%; margin-top:20px; border-collapse:collapse; text-align:center;'><tr><th>Method</th><th>AUC</th><th>Max SIC</th><th>Time</th></tr>"
            
            imgs = []
            for m in sorted(self.df['ID'].unique()):
                row = self.df[(self.df['ID']==m)&(self.df['MX']==mx)&(self.df['MY']==my)]
                bg = "#eafaf1" if m==base else "white"
                
                if row.empty:
                    html += f"<tr style='background:{bg}'><td>{m}</td><td colspan='3' style='color:#aaa'>No Data</td></tr>"
                    continue
                
                r = row.iloc[0]
                html += f"<tr style='background:{bg}'><td>{m}</td><td>{r['auc']:.4f}</td><td>{r['max_sic']:.2f}</td><td>{r['time']:.1f}</td></tr>"
                
                if r['png_path'] and os.path.exists(r['png_path']):
                    img_w = widgets.Image(value=open(r['png_path'], "rb").read(), format='png', width=300)
                    imgs.append(widgets.VBox([widgets.Label(m, style={'font-weight':'bold'}), img_w], layout=widgets.Layout(border='1px solid #eee', margin='5px')))

            html += "</table>"
            display(widgets.HTML(html))
            display(widgets.Box(imgs, layout=widgets.Layout(display='flex', flex_flow='row wrap')))

d = GridDashboard()
display(d.ui)

In [8]:
import os
import json
import glob
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import ipywidgets as widgets
import random
from IPython.display import display, Image, clear_output

# ==========================================
# 1. CONFIGURATION
# ==========================================
BASE_PATH = "/pscratch/sd/t/tihsu/database/GridStudy_v2/method/"
JSON_GRID_PATH = "config/sample_list.json" 
CUTFLOW_PATH = "/pscratch/sd/t/tihsu/database/GridStudy/cutflow.txt"

# ==========================================
# 2. DATA LOADER
# ==========================================

def load_reference_grid(json_path):
    mx_set, my_set = set(), set()
    if os.path.exists(json_path):
        with open(json_path, 'r') as f:
            config = json.load(f)
        for mass_set in config.get("signal", []):
            match = re.search(r"MX-([\d\.]+)_MY-([\d\.]+)", mass_set)
            if match:
                mx_set.add(int(float(match.group(1))))
                my_set.add(int(float(match.group(2))))
    return sorted(list(mx_set)), sorted(list(my_set))

def parse_mass_robust(text):
    match = re.search(r"MX-([\d\.]+).*?_MY-([\d\.]+)", text)
    if match:
        try:
            return int(float(match.group(1))), int(float(match.group(2)))
        except: return None, None
    return None, None

def load_cutflow_data(path):
    records = []
    if not os.path.exists(path):
        return pd.DataFrame()
        
    try:
        with open(path, 'r') as f:
            for line in f:
                if "MX-" not in line or "|" not in line: continue
                parts = [p.strip() for p in line.split('|')]
                if len(parts) >= 4:
                    mx, my = parse_mass_robust(parts[0])
                    if mx is not None:
                        records.append({
                            "MX": mx, 
                            "MY": my,
                            "passed": float(parts[2]),
                            "eff": float(parts[3])
                        })
    except Exception as e:
        print(f"Error parsing cutflow: {e}")
        
    return pd.DataFrame(records)

def load_grid_data(base_path):
    records = []

    def process_file(json_path, method, train_type):
        try:
            filename = os.path.basename(json_path)
            mx, my = parse_mass_robust(filename)
            if mx is None: mx, my = parse_mass_robust(os.path.dirname(json_path).split(os.sep)[-1])
            if mx is None: return

            if "eval_metric" not in json_path: return

            with open(json_path, 'r') as f: data = json.load(f)
            auc_val = data.get("auc", np.nan)
            if pd.isna(auc_val): return

            dirname = os.path.dirname(json_path)
            png_candidates = glob.glob(os.path.join(dirname, "*.png"))
            specific = [p for p in png_candidates if f"{mx}" in p and f"{my}" in p and "score" in p]
            png_path = specific[0] if specific else next((p for p in png_candidates if "score" in p), None)

            records.append({
                "Method": method, 
                "Type": train_type, 
                "ID": f"{method} ({train_type})",
                "MX": int(mx), "MY": int(my),
                "auc": auc_val,
                "auc_unc": data.get("auc_unc", 0.0),
                "max_sic": data.get("max_sic", np.nan), 
                "max_sic_unc": data.get("max_sic_unc", 0.0),
                "time": data.get("fitting_time", np.nan),
                "png_path": png_path
            })
        except Exception as e:
            return

    # 1. Individual
    for p in glob.glob(os.path.join(base_path, "*", "individual", "**", "*.json"), recursive=True):
        parts = p.split(os.sep)
        if "individual" in parts: 
            process_file(p, parts[parts.index("individual")-1], "individual")

    # 2. Parameterized
    for p in glob.glob(os.path.join(base_path, "*", "parameterized", "**", "*.json"), recursive=True):
        parts = p.split(os.sep)
        if "parameterized" in parts: 
            process_file(p, parts[parts.index("parameterized")-1], "parameterized")

    # 3. Sparse Parameterized
    sparse_pattern = os.path.join(base_path, "*", "parametrized_reduce_*", "All", "*.json")
    for p in glob.glob(sparse_pattern):
        parts = p.split(os.sep)
        try:
            method_name = parts[-4] 
            config_folder = parts[-3]
            factor_match = re.search(r"factor_x_(\d+)_y_(\d+)", config_folder)
            if factor_match:
                nx, ny = factor_match.groups()
                train_type = f"sparse_x{nx}y{ny}"
            else:
                train_type = "sparse_param"
            process_file(p, method_name, train_type)
        except IndexError:
            continue

    return pd.DataFrame(records)

# ==========================================
# 3. FANCY PLOTTING FUNCTION (CORRECTED Z-AXIS)
# ==========================================

def plot_comparison_fancy(df, df_cutflow, target_mx, target_my, all_mx, all_my, metric='max_sic', baseline_id='xgboost (individual)'):
    """
    Generates a 2D jitter plot integrating ALL mass points.
    - X-axis: Configuration (Param, Param 1/4, etc.)
    - Y-axis: Ratio to Baseline
    - Color (Z-axis): PASSED SAMPLE NUMBER (Blue/Red scheme)
    """
    
    # 1. PREPARE DATA (ALL POINTS)
    # ---------------------------------------------
    # Find baseline ID
    base_candidates = [m for m in df['ID'].unique() if 'xgboost' in m.lower() and 'individual' in m.lower()]
    real_baseline_id = base_candidates[0] if base_candidates else df['ID'].iloc[0]
    
    # Calculate Ratios
    base_df = df[df['ID'] == real_baseline_id].set_index(['MX', 'MY'])[metric]
    df_merged = df.join(base_df, on=['MX', 'MY'], rsuffix='_base')
    df_merged['ratio'] = df_merged[metric] / df_merged[f'{metric}_base']
    df_merged['ratio'] = df_merged['ratio'].fillna(0)
    
    # Merge with CUTFLOW to get 'passed' numbers for Z-axis coloring
    if not df_cutflow.empty:
        df_merged = pd.merge(df_merged, df_cutflow[['MX', 'MY', 'passed']], on=['MX', 'MY'], how='left')
        df_merged['passed'] = df_merged['passed'].fillna(0)
    else:
        df_merged['passed'] = 1 # Fallback

    # 2. DEFINE GROUPS & LABELS
    # ---------------------------------------------
    plot_data = []
    
    models = sorted(df_merged['Method'].unique(), key=lambda x: (0 if 'xgboost' in x.lower() else 1, x))
    
    for model in models:
        m_rows = df_merged[df_merged['Method'] == model]
        modes = sorted(m_rows['Type'].unique())
        
        group_modes = []
        for mode in modes:
            mode_rows = m_rows[m_rows['Type'] == mode]
            
            # --- LABEL MAPPING LOGIC ---
            # x1y1 -> Param
            # x2y2 -> Param (1/4)
            label = mode
            if 'individual' in mode.lower() or 'x1y1' in mode.lower():
                label = "Param"
            else:
                match = re.search(r"x(\d+)y(\d+)", mode)
                if match:
                    nx, ny = int(match.group(1)), int(match.group(2))
                    if nx == 2 and ny == 2:
                        label = "Param (1/4)"
                    else:
                        label = f"Param (1/{nx*ny})"
            
            # Collect points
            points_y = mode_rows['ratio'].values
            points_z = mode_rows['passed'].values # The color value
            
            group_modes.append({
                'mode_id': mode,
                'clean_label': label,
                'y': points_y,
                'z': points_z
            })
            
        plot_data.append({'model': model, 'modes': group_modes})

    # 3. PLOTTING
    # ---------------------------------------------
    fig, ax = plt.subplots(figsize=(12, 6), dpi=120)
    
    # Setup Colormap for Z-axis (Passed Numbers)
    # Using 'coolwarm' or 'RdBu_r' (Red=Low passed, Blue=High passed)
    cmap = plt.cm.RdBu 
    
    # Normalize color based on min/max of passed events across whole dataset
    min_pass = df_merged['passed'].min()
    max_pass = df_merged['passed'].max()
    norm = plt.Normalize(vmin=min_pass, vmax=max_pass)

    x_cursor = 0
    x_ticks = []
    x_labels = []
    
    jitter_width = 0.2
    
    for group in plot_data:
        model_name = group['model']
        modes = group['modes']
        
        start_x = x_cursor
        
        for m_obj in modes:
            y_vals = m_obj['y']
            z_vals = m_obj['z'] # Color data
            label = m_obj['clean_label']
            
            if len(y_vals) > 0:
                # Jitter X
                noise = np.random.uniform(-jitter_width, jitter_width, size=len(y_vals))
                x_vals = x_cursor + noise
                
                # Scatter Plot
                # c = z_vals (Passed Number)
                sc = ax.scatter(x_vals, y_vals, 
                           c=z_vals,        
                           cmap=cmap, 
                           norm=norm,
                           s=25,            
                           alpha=0.7,       
                           edgecolors='grey',
                           linewidth=0.3,
                           zorder=2)
            
            x_ticks.append(x_cursor)
            x_labels.append(label)
            x_cursor += 1
            
        # Group Formatting
        end_x = x_cursor - 1
        center_x = (start_x + end_x) / 2
        
        # MODEL NAME (Middle Top, Bold)
        ax.text(center_x, 1.45, model_name.upper(), 
                ha='center', va='bottom', fontweight='bold', fontsize=12, color='#333')
        
        # Vertical separator
        ax.axvline(x=x_cursor - 0.5, color='#eee', linestyle='-', linewidth=1.5, zorder=0)
        x_cursor += 0.5

    # 4. FINAL AESTHETICS
    ax.axhline(1.0, color='gray', linestyle='--', linewidth=1, alpha=0.5, zorder=1)
    
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=10)
    
    ax.set_ylabel(f"Ratio to Baseline ({metric})", fontsize=11)
    
    # Remove top/right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Colorbar -> Shows Passed Number
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, aspect=30, pad=0.01)
    cbar.set_label('Passed Events (Sample Size)', rotation=270, labelpad=15)

    plt.title(f"Performance vs Architecture (Color = Sample Size)", loc='left', fontsize=12, pad=30)
    plt.ylim(0.5, 1.6) 
    
    plt.tight_layout()
    plt.show()

# ==========================================
# 4. DASHBOARD CLASS
# ==========================================

class GridDashboard:
    def __init__(self):
        self.df = pd.DataFrame()
        self.df_cutflow = pd.DataFrame()
        self.expected_mx = []
        self.expected_my = []
        
        # UI Elements
        self.btn_load = widgets.Button(description="Refresh Data", button_style='primary', icon='refresh')
        
        self.dd_metric = widgets.Dropdown(
            description="Metric:", 
            options=[('AUC', 'auc'), ('Max SIC', 'max_sic'), ("Time", "time"), ('Cutflow: Passed', 'passed'), ('Cutflow: Eff %', 'eff')], 
            value='auc'
        )
        
        self.tog_compare = widgets.ToggleButtons(options=['Ratio', 'Pull'], description='Compare:', button_style='')
        self.cb_show_text = widgets.Checkbox(value=True, description='Values')
        self.slider_font = widgets.IntSlider(value=10, min=6, max=18, description='Font:')
        self.slider_prec = widgets.IntSlider(value=3, min=0, max=5, description='Decimals:')
        
        self.dd_baseline = widgets.Dropdown(description="Baseline:")
        self.dd_mx = widgets.Dropdown(description="MX:")
        self.dd_my = widgets.Dropdown(description="MY:")
        
        self.out_status = widgets.Output()
        self.out_heatmap = widgets.Output()
        self.out_details = widgets.Output()
        
        # Events
        self.btn_load.on_click(self.load_data)
        for w in [self.dd_metric, self.dd_baseline, self.tog_compare, self.cb_show_text, self.slider_font, self.slider_prec]:
            w.observe(self.render_plots, names='value')
        self.dd_mx.observe(self.update_my_options, names='value')
        self.dd_my.observe(self.render_details, names='value')

        self.ui = widgets.VBox([
            widgets.HBox([self.btn_load, self.dd_metric, self.tog_compare]),
            widgets.HBox([self.dd_baseline, self.cb_show_text, self.slider_font, self.slider_prec]),
            self.out_status, widgets.HTML("<hr>"), self.out_heatmap,
            widgets.HTML("<hr><h3>Drill Down Inspector</h3>"), widgets.HBox([self.dd_mx, self.dd_my]), self.out_details
        ])
        self.load_data(None)

    def load_data(self, b):
        with self.out_status:
            clear_output()
            print("Loading...")
            ref_mx, ref_my = load_reference_grid(JSON_GRID_PATH)
            
            self.df_cutflow = load_cutflow_data(CUTFLOW_PATH)
            
            df_raw = load_grid_data(BASE_PATH)
            
            if not df_raw.empty:
                self.df = df_raw.sort_values('auc', ascending=False).drop_duplicates(subset=['ID', 'MX', 'MY'], keep='first')
                self.expected_mx = sorted(list(set(ref_mx) | set(self.df['MX'].unique())))
                self.expected_my = sorted(list(set(ref_my) | set(self.df['MY'].unique())))
                print(f"Loaded {len(self.df)} records.")
            else:
                self.df = pd.DataFrame(columns=['ID', 'MX', 'MY', 'auc'])
                print("No data found.")

        if not self.df.empty:
            methods = sorted(self.df['ID'].unique())
            self.dd_baseline.options = methods
            if methods: self.dd_baseline.value = next((m for m in methods if 'xgboost' in m.lower() and 'individual' in m.lower()), methods[0])
            self.dd_mx.options = self.expected_mx
            if self.expected_mx: 
                self.dd_mx.value = self.expected_mx[0]
                self.update_my_options(None)
        
        self.render_plots(None)

    def update_my_options(self, change):
        self.dd_my.options = self.expected_my
        if self.expected_my: self.dd_my.value = self.expected_my[0]
        self.render_details(None)

    def get_pivot(self, method, metric):
        sub = self.df[self.df['ID'] == method]
        if sub.empty: return pd.DataFrame(index=self.expected_my, columns=self.expected_mx)
        return sub.pivot_table(index='MY', columns='MX', values=metric, aggfunc='mean').reindex(index=self.expected_my, columns=self.expected_mx).sort_index(ascending=False)

    def render_plots(self, change):
        if not self.expected_mx: return
        with self.out_heatmap:
            clear_output(wait=True)
            met = self.dd_metric.value
            if met in ['passed', 'eff']:
                if self.df_cutflow.empty:
                    print("No Cutflow data.")
                    return
                pivot = self.df_cutflow.pivot(index='MY', columns='MX', values=met).reindex(index=self.expected_my, columns=self.expected_mx).sort_index(ascending=False)
                fig, ax = plt.subplots(figsize=(10, 8), dpi=100)
                sns.heatmap(pivot, ax=ax, annot=self.cb_show_text.value, fmt=".0f" if met=='passed' else ".2f", cmap="viridis", cbar=True)
                plt.show()
                return

            if not self.dd_baseline.value: return
            base = self.dd_baseline.value
            methods = sorted(self.df['ID'].unique())
            n = len(methods)
            fig, axes = plt.subplots(n, 2, figsize=(16, 5*n), dpi=100)
            if n==1: axes = axes.reshape(1, -1)
            
            p_base = self.get_pivot(base, met)
            g_min, g_max = self.df[met].min(), self.df[met].max()

            for i, m in enumerate(methods):
                p_curr = self.get_pivot(m, met)
                sns.heatmap(p_curr, ax=axes[i,0], annot=self.cb_show_text.value, fmt=f".{self.slider_prec.value}f", cmap="PuBu", vmin=g_min, vmax=g_max, cbar=True)
                axes[i,0].set_title(f"{m}", fontweight='bold')
                axes[i,0].set_ylabel("MY")
                
                ax_c = axes[i,1]
                if self.tog_compare.value == 'Ratio':
                    sns.heatmap(p_curr/p_base, ax=ax_c, annot=self.cb_show_text.value, fmt=f".{self.slider_prec.value}f", cmap="RdBu_r", center=1, vmin=0.5, vmax=1.5)
                    ax_c.set_title("Ratio")
                else:
                    sns.heatmap(p_curr - p_base, ax=ax_c, annot=self.cb_show_text.value, fmt=".2f", cmap="RdBu_r", center=0)
                    ax_c.set_title("Diff")
                axes[i,0].set_xlabel("MX" if i==n-1 else "")
                ax_c.set_xlabel("MX" if i==n-1 else "")
                ax_c.set_yticks([])

            plt.tight_layout()
            plt.show()

    def render_details(self, change):
        mx, my = self.dd_mx.value, self.dd_my.value
        base = self.dd_baseline.value
        if not mx or not my: return
        with self.out_details:
            clear_output(wait=True)
            
            cut_val = "N/A"
            if not self.df_cutflow.empty:
                subset = self.df_cutflow[(self.df_cutflow['MX']==mx) & (self.df_cutflow['MY']==my)]
                if not subset.empty:
                    cut_val = f"{int(subset.iloc[0]['passed'])} (Eff: {subset.iloc[0]['eff']}%)"
            
            display(widgets.HTML(f"""
            <div style="background-color: #e3f2fd; border-left: 6px solid #2196f3; padding: 10px; margin-bottom: 20px;">
                <h4 style="margin:0; color:#0d47a1;">Signal Point: MX-{mx} MY-{my}</h4>
                Passed Events: <b>{cut_val}</b>
            </div>
            """))

            plot_metric = 'max_sic' if 'sic' in self.dd_metric.value else 'auc'
            plot_base = next((m for m in self.df['ID'].unique() if 'xgboost' in m.lower() and 'individual' in m.lower()), base)
            
            try:
                # UPDATED: Plotting function now handles data integration internally
                plot_comparison_fancy(self.df, self.df_cutflow, mx, my, self.expected_mx, self.expected_my, metric=plot_metric, baseline_id=plot_base)
            except Exception as e:
                print(f"Could not render summary plot: {e}")
                import traceback
                traceback.print_exc()

            html = "<table border='1' style='width:100%; margin-top:20px; border-collapse:collapse; text-align:center;'><tr><th>Method</th><th>AUC</th><th>Max SIC</th><th>Time</th></tr>"
            
            imgs = []
            for m in sorted(self.df['ID'].unique()):
                row = self.df[(self.df['ID']==m)&(self.df['MX']==mx)&(self.df['MY']==my)]
                bg = "#eafaf1" if m==base else "white"
                
                if row.empty:
                    html += f"<tr style='background:{bg}'><td>{m}</td><td colspan='3' style='color:#aaa'>No Data</td></tr>"
                    continue
                
                r = row.iloc[0]
                html += f"<tr style='background:{bg}'><td>{m}</td><td>{r['auc']:.4f}</td><td>{r['max_sic']:.2f}</td><td>{r['time']:.1f}</td></tr>"
                
                if r['png_path'] and os.path.exists(r['png_path']):
                    img_w = widgets.Image(value=open(r['png_path'], "rb").read(), format='png', width=300)
                    imgs.append(widgets.VBox([widgets.Label(m, style={'font-weight':'bold'}), img_w], layout=widgets.Layout(border='1px solid #eee', margin='5px')))

            html += "</table>"
            display(widgets.HTML(html))
            display(widgets.Box(imgs, layout=widgets.Layout(display='flex', flex_flow='row wrap')))

d = GridDashboard()
display(d.ui)

In [20]:
import os
import json
import glob
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

# ==========================================
# 1. CONFIGURATION
# ==========================================
BASE_PATH = "/pscratch/sd/t/tihsu/database/GridStudy_v2/method/"
JSON_GRID_PATH = "config/sample_list.json" 
CUTFLOW_PATH = "/pscratch/sd/t/tihsu/database/GridStudy/cutflow.txt"

# ==========================================
# 2. DATA LOADER
# ==========================================

def load_reference_grid(json_path):
    mx_set, my_set = set(), set()
    if os.path.exists(json_path):
        with open(json_path, 'r') as f:
            config = json.load(f)
        for mass_set in config.get("signal", []):
            match = re.search(r"MX-([\d\.]+)_MY-([\d\.]+)", mass_set)
            if match:
                mx_set.add(int(float(match.group(1))))
                my_set.add(int(float(match.group(2))))
    return sorted(list(mx_set)), sorted(list(my_set))

def parse_mass_robust(text):
    match = re.search(r"MX-([\d\.]+).*?_MY-([\d\.]+)", text)
    if match:
        try:
            return int(float(match.group(1))), int(float(match.group(2)))
        except: return None, None
    return None, None

def load_cutflow_data(path):
    records = []
    if not os.path.exists(path):
        return pd.DataFrame()
    try:
        with open(path, 'r') as f:
            for line in f:
                if "MX-" not in line or "|" not in line: continue
                parts = [p.strip() for p in line.split('|')]
                if len(parts) >= 4:
                    mx, my = parse_mass_robust(parts[0])
                    if mx is not None:
                        records.append({"MX": mx, "MY": my, "passed": float(parts[2]), "eff": float(parts[3])})
    except Exception as e:
        print(f"Error parsing cutflow: {e}")
    return pd.DataFrame(records)

def load_grid_data(base_path):
    records = []
    def process_file(json_path, method, train_type):
        try:
            filename = os.path.basename(json_path)
            mx, my = parse_mass_robust(filename)
            if mx is None: mx, my = parse_mass_robust(os.path.dirname(json_path).split(os.sep)[-1])
            if mx is None: return
            if "eval_metric" not in json_path: return
            with open(json_path, 'r') as f: data = json.load(f)
            auc_val = data.get("auc", np.nan)
            if pd.isna(auc_val): return
            
            dirname = os.path.dirname(json_path)
            png_candidates = glob.glob(os.path.join(dirname, "*.png"))
            specific = [p for p in png_candidates if f"{mx}" in p and f"{my}" in p and "score" in p]
            png_path = specific[0] if specific else next((p for p in png_candidates if "score" in p), None)

            records.append({
                "Method": method, "Type": train_type, "ID": f"{method} ({train_type})",
                "MX": int(mx), "MY": int(my),
                "auc": auc_val, "max_sic": data.get("max_sic", np.nan),
                "time": data.get("fitting_time", np.nan), "png_path": png_path
            })
        except: return

    # Standardize path finding
    for root, dirs, files in os.walk(base_path):
        for file in files:
            if file.endswith(".json") and "eval_metric" in file:
                full_path = os.path.join(root, file)
                if "individual" in full_path:
                    process_file(full_path, full_path.split(os.sep)[-4], "individual")
                elif "parameterized" in full_path and "reduce" not in full_path:
                    process_file(full_path, full_path.split(os.sep)[-4], "parameterized")
                elif "parametrized_reduce" in full_path:
                    try:
                        parts = full_path.split(os.sep)
                        method = parts[-4]
                        cfg = parts[-3]
                        mat = re.search(r"factor_x_(\d+)_y_(\d+)", cfg)
                        if mat: process_file(full_path, method, f"sparse_x{mat.group(1)}y{mat.group(2)}")
                        else: process_file(full_path, method, "sparse_param")
                    except: continue
    return pd.DataFrame(records)

# ==========================================
# 3. FANCY PLOTTING FUNCTION (FINAL)
# ==========================================
def plot_comparison_fancy(df, df_cutflow, target_mx, target_my, all_mx, all_my, metric='max_sic', baseline_id=None):
    """
    Comparison plot with:
    - Corrected baseline logic (respects dropdown).
    - Headers pinned to top frame.
    - 'Fraction > Baseline' statistic elegantly placed at the bottom.
    """
    # 1. Baseline & Ratios
    real_baseline_id = baseline_id if baseline_id else df['ID'].iloc[0]
    
    if real_baseline_id not in df['ID'].values:
        print(f"Warning: Baseline '{real_baseline_id}' not found. Defaulting to first method.")
        real_baseline_id = df['ID'].iloc[0]

    base_df = df[df['ID'] == real_baseline_id].set_index(['MX', 'MY'])[metric]
    df_merged = df.join(base_df, on=['MX', 'MY'], rsuffix='_base')
    df_merged['ratio'] = df_merged[metric] / df_merged[f'{metric}_base']
    df_merged['ratio'] = df_merged['ratio'].fillna(0)

    # Merge Passed Events for Color
    if not df_cutflow.empty:
        df_merged = pd.merge(df_merged, df_cutflow[['MX', 'MY', 'passed']], on=['MX', 'MY'], how='left').fillna(0)
    else:
        df_merged['passed'] = 1

    # 2. Logic & Grouping
    plot_data = []
    models = sorted(df_merged['Method'].unique(), key=lambda x: (0 if 'xgboost' in x.lower() else 1, x))
    
    for model in models:
        m_rows = df_merged[df_merged['Method'] == model]
        modes = sorted(m_rows['Type'].unique())
        
        group_modes = []
        for mode in modes:
            mode_rows = m_rows[m_rows['Type'] == mode]
            
            # --- Labeling Logic ---
            label = mode
            step_x, step_y = 1, 1
            
            if 'individual' in mode.lower():
                label = "Individual"
            elif 'x1y1' in mode.lower():
                label = "Param(Full)"
            else:
                match = re.search(r"x(\d+)y(\d+)", mode)
                if match:
                    step_x, step_y = int(match.group(1)), int(match.group(2))
                    if step_x == 2 and step_y == 2: 
                        label = "Param (1/4)"
                    else: 
                        label = f"Param (1/{step_x*step_y})"
                elif 'param' in mode.lower():
                    label = "Param"

            # Determine Trained vs Interpolated
            points_data = []
            for _, r in mode_rows.iterrows():
                try:
                    idx_x = all_mx.index(r['MX'])
                    idx_y = all_my.index(r['MY'])
                    is_trained = (idx_x % step_x == 0) and (idx_y % step_y == 0)
                except ValueError:
                    is_trained = False
                
                points_data.append({
                    'y': r['ratio'],
                    'z': r['passed'],
                    'is_trained': is_trained
                })

            group_modes.append({'clean_label': label, 'data': points_data})
        plot_data.append({'model': model, 'modes': group_modes})

    # 3. Plotting
    fig, ax = plt.subplots(figsize=(14, 7), dpi=120) # Slightly wider for elegance
    cmap = plt.cm.RdBu 
    norm = plt.Normalize(vmin=df_merged['passed'].min(), vmax=df_merged['passed'].max())
    
    x_cursor = 0
    x_ticks, x_labels = [], []
    jitter_width = 0.2

    for group in plot_data:
        model_name = group['model']
        start_x = x_cursor
        
        for m_obj in group['modes']:
            data = m_obj['data']
            label = m_obj['clean_label']
            
            if data:
                y_vals = np.array([d['y'] for d in data])
                z_vals = np.array([d['z'] for d in data])
                trained_mask = np.array([d['is_trained'] for d in data])
                
                # --- STATS CALCULATION ---
                # Calculate fraction better than baseline (> 1.0)
                n_total = len(y_vals)
                if n_total > 0:
                    frac_better = np.sum(y_vals > 1.0) / n_total
                    stat_text = f"{frac_better:.1%} > Base."

                    stat_color = '#4A6C44' if frac_better >= 0.5 else '#A35648'
                else:
                    stat_text = "N/A"
                    stat_color = 'gray'

                # Jitter X
                noise = np.random.uniform(-jitter_width, jitter_width, size=len(y_vals))
                x_vals = x_cursor + noise
                
                # Plot Points
                if np.any(trained_mask):
                    ax.scatter(x_vals[trained_mask], y_vals[trained_mask], c=z_vals[trained_mask], 
                               cmap=cmap, norm=norm, s=40, marker='D', edgecolors='k', linewidth=0.5, zorder=3)
                if np.any(~trained_mask):
                    ax.scatter(x_vals[~trained_mask], y_vals[~trained_mask], c=z_vals[~trained_mask], 
                               cmap=cmap, norm=norm, s=25, marker='o', edgecolors='k', linewidth=0.3, alpha=0.6, zorder=2)
                
                # --- ADD STAT TEXT ---
                # Placing it at y=0.1 (Data Coordinates) uses the empty bottom space effectively
                ax.text(x_cursor, 0.1, stat_text, 
                        ha='center', va='center', fontsize=6,fontweight='bold', 
                        color=stat_color, bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7))

            x_ticks.append(x_cursor)
            x_labels.append(label)
            x_cursor += 1
            
        # Headers (Model Names)
        end_x = x_cursor - 1
        center_x = (start_x + end_x) / 2
        
        # Header text relative to frame (always just above top axis)
        ax.text(center_x, 1.02, model_name.upper(), 
                ha='center', va='bottom', fontweight='bold', fontsize=12, color='#333',
                transform=ax.get_xaxis_transform()) 
        
        ax.axvline(x=x_cursor - 0.5, color='#eee', linestyle='-', linewidth=1.5, zorder=0)
        x_cursor += 0.5

    # 4. Legend & Style
    ax.axhline(1.0, color='gray', linestyle='--', linewidth=1, alpha=0.5, zorder=1)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=10)
    ax.set_ylabel(f"Ratio to Baseline ({metric})", fontsize=11)
    
    # Clean spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False) # Optional: cleaner look
    ax.tick_params(axis='y', length=0)   # Optional: remove y-ticks for cleaner look
    ax.grid(axis='y', linestyle=':', alpha=0.4, zorder=0) # Add light grid
    
    # Colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, aspect=30, pad=0.01)
    cbar.set_label('Passed Events (Sample Size)', rotation=270, labelpad=15)

    plt.ylim(0.0, 3) # User specified range

    # Legend
    legend_handles = [
        mlines.Line2D([], [], color='white', marker='D', markerfacecolor='gray', markeredgecolor='k', markersize=8, label='Trained Point'),
        mlines.Line2D([], [], color='white', marker='o', markerfacecolor='gray', markeredgecolor='k', markersize=8, label='Interpolated')
    ]
    ax.legend(handles=legend_handles, loc='upper left', frameon=False, fontsize=10, bbox_to_anchor=(0, 1.0))

    plt.tight_layout()
    plt.show()

# ==========================================
# 4. DASHBOARD CLASS
# ==========================================

class GridDashboard:
    def __init__(self):
        self.df = pd.DataFrame()
        self.df_cutflow = pd.DataFrame()
        self.expected_mx, self.expected_my = [], []
        
        self.btn_load = widgets.Button(description="Refresh", button_style='primary')
        self.dd_metric = widgets.Dropdown(options=[('AUC', 'auc'), ('Max SIC', 'max_sic')], value='max_sic', description='Metric:')
        self.dd_baseline = widgets.Dropdown(description="Baseline:")
        self.dd_mx = widgets.Dropdown(description="MX:")
        self.dd_my = widgets.Dropdown(description="MY:")
        self.out_plot = widgets.Output()
        
        self.btn_load.on_click(self.load_data)
        self.dd_metric.observe(self.render, names='value')
        self.dd_baseline.observe(self.render, names='value')
        self.dd_mx.observe(self.render, names='value')
        
        self.ui = widgets.VBox([
            widgets.HBox([self.btn_load, self.dd_metric, self.dd_baseline]),
            widgets.HBox([self.dd_mx, self.dd_my]),
            self.out_plot
        ])
        self.load_data(None)

    def load_data(self, b):
        ref_mx, ref_my = load_reference_grid(JSON_GRID_PATH)
        self.df_cutflow = load_cutflow_data(CUTFLOW_PATH)
        self.df = load_grid_data(BASE_PATH)
        
        if not self.df.empty:
            self.df = self.df.sort_values('auc', ascending=False).drop_duplicates(subset=['ID','MX','MY'])
            self.expected_mx = sorted(list(set(ref_mx)|set(self.df['MX'])))
            self.expected_my = sorted(list(set(ref_my)|set(self.df['MY'])))
            
            methods = sorted(self.df['ID'].unique())
            self.dd_baseline.options = methods
            if methods: self.dd_baseline.value = methods[0]
            self.dd_mx.options = self.expected_mx
            self.dd_my.options = self.expected_my
            
        self.render(None)

    def render(self, b):
        with self.out_plot:
            clear_output(wait=True)
            if self.df.empty: return
            
            mx, my = self.dd_mx.value, self.dd_my.value
            
            plot_comparison_fancy(
                self.df, self.df_cutflow, 
                mx, my,
                self.expected_mx, self.expected_my,
                metric=self.dd_metric.value,
                baseline_id=self.dd_baseline.value
            )

d = GridDashboard()
display(d.ui)